# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
docs[:3]

[Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 0}, page_content='NBER WORKING PAPER SERIES\nHOW PEOPLE USE CHATGPT\nAaron Chatterji\nThomas Cunningham\nDavid J. Deming\nZoe Hitzig\nChristopher Ong\nCarl Yan Shan\nKevin Wadman\nWorking Paper 34255\nhttp://www.nber.org/papers/w34255\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nSeptember 2025\nWe acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan \nBeiermeister, Rachel Brown, Cassandra Duchan Solis, Jason Kwon,

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '05a838'. Skipping!
Property 'summary' already exists in node '840183'. Skipping!
Property 'summary' already exists in node 'a1adce'. Skipping!
Property 'summary' already exists in node 'f22fa2'. Skipping!
Property 'summary' already exists in node '8f9b68'. Skipping!
Property 'summary' already exists in node 'c4d9ac'. Skipping!
Property 'summary' already exists in node 'a87f8d'. Skipping!
Property 'summary' already exists in node '3c8501'. Skipping!
Property 'summary' already exists in node '114c36'. Skipping!
Property 'summary' already exists in node '84d223'. Skipping!
Property 'summary' already exists in node '3c78ef'. Skipping!
Property 'summary' already exists in node '97c37f'. Skipping!
Property 'summary' already exists in node '3052cc'. Skipping!
Property 'summary' already exists in node '8f743a'. Skipping!
Property 'summary' already exists in node '37402a'. Skipping!
Property 'summary' already exists in node '4d9c4e'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '97c37f'. Skipping!
Property 'summary_embedding' already exists in node '05a838'. Skipping!
Property 'summary_embedding' already exists in node '840183'. Skipping!
Property 'summary_embedding' already exists in node '8f9b68'. Skipping!
Property 'summary_embedding' already exists in node 'a1adce'. Skipping!
Property 'summary_embedding' already exists in node 'f22fa2'. Skipping!
Property 'summary_embedding' already exists in node 'e9ad26'. Skipping!
Property 'summary_embedding' already exists in node 'a87f8d'. Skipping!
Property 'summary_embedding' already exists in node '3c78ef'. Skipping!
Property 'summary_embedding' already exists in node '84d223'. Skipping!
Property 'summary_embedding' already exists in node '114c36'. Skipping!
Property 'summary_embedding' already exists in node '3c8501'. Skipping!
Property 'summary_embedding' already exists in node '3052cc'. Skipping!
Property 'summary_embedding' already exists in node '8f743a'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 713)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 713)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

1. SingleHopSpecificQuerySynthesizer: 

Generates simple questions that require retrieving from a single node.

2. MultiHopSpecificQuerySynthesizer: 

Generates queries that require retrieving and combining multiple related nodes. They reference specific entities in the nodes.

3. MultiHopAbstractQuerySynthesizer: 

Uses multiple nodes to genarate query. Queries ask about abstract relationships or “themes”.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How does Ling relate to ChatGPT usage in work ...,[Introduction ChatGPT launched in November 202...,The provided context does not mention Ling or ...,single_hop_specifc_query_synthesizer
1,how much chatgpt used work or not work,[Month Non-Work (M) (%) Work (M) (%) Total Mes...,"In June 2024, ChatGPT messages were 53% non-wo...",single_hop_specifc_query_synthesizer
2,How does Collis relate to the analysis of AI u...,[Total daily counts are exact measurements of ...,The context mentions Collis in relation to stu...,single_hop_specifc_query_synthesizer
3,What information does Appendix D provide regar...,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
4,"What does the 29,000 messages per second refer...",[Conclusion This paper studies the rapid growt...,"The 29,000 messages per second refer to the ra...",single_hop_specifc_query_synthesizer
5,How does the classification of work-related me...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The context shows that while total message cou...,multi_hop_abstract_query_synthesizer
6,How does the rapid growth of consumer ChatGPT ...,[<1-hop>\n\nTotal daily counts are exact measu...,The context indicates that total daily message...,multi_hop_abstract_query_synthesizer
7,how message counts from june 2024 to june 2025...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that total daily message co...,multi_hop_abstract_query_synthesizer
8,wHAT is the mONTH in 2025 wHEN ChatGPT had moR...,[<1-hop>\n\nConclusion This paper studies the ...,"In July 2025, ChatGPT had a higher percentage ...",multi_hop_specific_query_synthesizer
9,"In Jun 2024 and Jun 2025, how non-work message...",[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In June 2024, non-work messages made up 53% of...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Property 'summary' already exists in node '4cb1ef'. Skipping!
Property 'summary' already exists in node '0e657a'. Skipping!
Property 'summary' already exists in node 'b2cd46'. Skipping!
Property 'summary' already exists in node '97e9da'. Skipping!
Property 'summary' already exists in node '03680f'. Skipping!
Property 'summary' already exists in node '103c76'. Skipping!
Property 'summary' already exists in node 'bae297'. Skipping!
Property 'summary' already exists in node '3558bb'. Skipping!
Property 'summary' already exists in node '0b8b11'. Skipping!
Property 'summary' already exists in node '0a5850'. Skipping!
Property 'summary' already exists in node 'ed8694'. Skipping!
Property 'summary' already exists in node '150ff8'. Skipping!
Property 'summary' already exists in node 'f9cb79'. Skipping!
Property 'summary' already exists in node '2f833d'. Skipping!
Property 'summary' already exists in node '638caf'. Skipping!
Property 'summary' already exists in node '9c23a3'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '03680f'. Skipping!
Property 'summary_embedding' already exists in node 'bae297'. Skipping!
Property 'summary_embedding' already exists in node '97e9da'. Skipping!
Property 'summary_embedding' already exists in node '103c76'. Skipping!
Property 'summary_embedding' already exists in node '0e657a'. Skipping!
Property 'summary_embedding' already exists in node '4cb1ef'. Skipping!
Property 'summary_embedding' already exists in node '0a5850'. Skipping!
Property 'summary_embedding' already exists in node 'ed8694'. Skipping!
Property 'summary_embedding' already exists in node 'b2cd46'. Skipping!
Property 'summary_embedding' already exists in node '0b8b11'. Skipping!
Property 'summary_embedding' already exists in node '3558bb'. Skipping!
Property 'summary_embedding' already exists in node '638caf'. Skipping!
Property 'summary_embedding' already exists in node '2f833d'. Skipping!
Property 'summary_embedding' already exists in node '150ff8'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How many users are using ChatGPT like the 700 ...,[Introduction ChatGPT launched in November 202...,"By July 2025, 700 million users were sending a...",single_hop_specifc_query_synthesizer
1,Korinek and Suh 2024 what they say about AI an...,[Introduction ChatGPT launched in November 202...,"The context mentions that Korinek and Suh, 202...",single_hop_specifc_query_synthesizer
2,What significance does June 2025 hold in the c...,[Table 1: ChatGPT daily message counts (millio...,"The context reports data ending on June 26, 20...",single_hop_specifc_query_synthesizer
3,What is the significance of June 2024 in the c...,[Table 1: ChatGPT daily message counts (millio...,"The report provides data ending on June 26th, ...",single_hop_specifc_query_synthesizer
4,SOC2 codes 13 what are they and how they relat...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 reports resu...,single_hop_specifc_query_synthesizer
5,What are SOC2 codes 19?,[Variation by Occupation Figure 23 presents va...,The context does not provide a specific defini...,single_hop_specifc_query_synthesizer
6,Wht work activites and mesage distrubtion by o...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data presents variation in ChatGPT usage b...,multi_hop_abstract_query_synthesizer
7,How does the launch and rapid adoption of Chat...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT was launched in November 2022 and, by ...",multi_hop_abstract_query_synthesizer
8,How do user behavior and usage patterns of Cha...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The rapid adoption of ChatGPT, with 18 billion...",multi_hop_abstract_query_synthesizer
9,How do LLMs like ChatGPT's AI capabiltys help ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, based on Large Language Models (LLMs)...",multi_hop_abstract_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside work. AI is employed to perform workplace tasks either by augmenting or automating human labor. Users engage with AI to produce writing, software code, spreadsheets, and other digital products, which distinguishes generative AI from traditional technologies like web search engines. Conversations with AI can be classified by user intent as Asking (seeking information or advice), Doing (producing or completing tasks), or Expressing (self-expression or interaction).\n\nAdditionally, generative AI is highly flexible and used for tasks ranging from professional occupations requiring problem-solving assistance to personal uses such as relationships, personal reflection, games, and role play. Overall, AI serves both as a co-worker producing output and as a co-pilot that gives advice to improve human productivity.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: corectness
- `labeled_helpfulness_evaluator`: helpfulness 
- `dopeness_evaluator`: style and creativity



## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'giving-button-54' at:
https://smith.langchain.com/o/38ed95dd-0f48-42f3-8a8b-a75a23f461eb/datasets/88d68d12-5fe8-4b25-a758-53012237e8dd/compare?selectedSessions=086c069f-99d1-4ea8-beb1-8fed985db240




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on the message volume comparison, how ha...",The growth in non-work ChatGPT messages has in...,None,The data shows that non-work messages have gro...,1,1,0,3.872764,70824b8f-d252-41e4-bc55-e323c4aca735,91de0a3a-e8ef-4956-99e7-816c4a1311a1
1,How do LLMs like ChatGPT's AI capabiltys help ...,Based on the context provided:\n\nLLMs like Ch...,None,"ChatGPT, based on Large Language Models (LLMs)...",1,1,0,6.766391,bb8953d8-6be4-4177-9417-d1861ccc43e0,a73ed5be-0b6e-4538-a9aa-14899ab98997
2,How do user behavior and usage patterns of Cha...,"Based on the provided context, user behavior a...",None,"The rapid adoption of ChatGPT, with 18 billion...",1,1,0,5.290649,9bfea33d-03a6-48d5-9f31-93c9d3f1b046,a48eef4b-181c-4290-bec3-bce98fd2572f
3,How does the launch and rapid adoption of Chat...,The launch and rapid adoption of ChatGPT illus...,None,"ChatGPT was launched in November 2022 and, by ...",1,1,0,3.552954,d7f76581-4410-41ef-aa78-96e2bb663040,16313230-a60b-408b-b45f-1e5484d7998f
4,Wht work activites and mesage distrubtion by o...,The data maps message content to work activiti...,None,The data presents variation in ChatGPT usage b...,1,1,0,6.969252,eee0be09-86c6-4ff3-ae12-86823fab718c,adf7e54c-ed63-403e-8704-5244a39575c3
5,What are SOC2 codes 19?,"Based on the provided context, SOC2 code 19 co...",None,The context does not provide a specific defini...,0,0,0,1.454771,f61ed56f-a224-43a2-b63e-baeef398b0ce,b4fc5aea-ad6a-46db-acca-b53125dcdd6e
6,SOC2 codes 13 what are they and how they relat...,"Based on the provided context, SOC2 code 13 co...",None,Variation by Occupation Figure 23 reports resu...,1,1,0,4.181283,bd773135-76a2-44a7-98f8-2fd225a9348b,b265b920-ee11-46fe-a464-e17aa0fcbd99
7,What is the significance of June 2024 in the c...,The significance of June 2024 in the context o...,None,"The report provides data ending on June 26th, ...",0,0,0,2.444369,70771476-d2e2-45af-abbf-591f9768deff,1d842cf1-417b-40e2-ad5b-ffd0093a0972
8,What significance does June 2025 hold in the c...,June 2025 marks a significant point in ChatGPT...,None,"The context reports data ending on June 26, 20...",1,1,0,3.113730,116483b7-96d1-4116-8176-6e01f58c1588,2739889f-442f-45ae-bfad-df5044d38ec8
9,Korinek and Suh 2024 what they say about AI an...,"Based on the provided context, Korinek and Don...",None,"The context mentions that Korinek and Suh, 202...",0,0,0,3.640563,82d17e62-0616-402a-b242-eaa72d0b606b,22f7a8bd-6dc7-4346-93e7-bb704cf9d48b


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

Increasing chunk size allows llm see more content. 



In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

Larger embedding model capture semantic meaning more accurately.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, here’s the lowdown on the AI-money hustle straight from the dopest brainwaves in that context: \n\nPeople ain’t just getting AI to punch clock tasks — they’re leveling up their *decision-making* game with ChatGPT acting as a slick advisor and research sidekick. This means knowledge workers, whose jackpot is smart choices and sharp insights, are *boosting their productivity* by having AI help them make calls that count.\n\nThe magic sauce? AI’s flex isn’t just automation—it’s *decision support* that hones the quality of work output. So instead of AI replacing jobs outright, it’s like a power-up making pros smarter, faster, and more effective in their cash-generating gigs. \n\nPlus, with a mind-blowing $97 billion+ surplus just in the US from people valuing this AI-fueled boost, the message is loud and clear: AI’s cash game is all about *augmenting human smarts* to rake in more dough through better work, not just grinding tasks on autopilot.\n\nIn short: AI is the dopest wingma

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'worthwhile-game-58' at:
https://smith.langchain.com/o/38ed95dd-0f48-42f3-8a8b-a75a23f461eb/datasets/88d68d12-5fe8-4b25-a758-53012237e8dd/compare?selectedSessions=6cd0e23d-6a93-4e63-a3f4-b3b6af080626




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on the message volume comparison, how ha...","Yo, check this out—non-work ChatGPT messages a...",None,The data shows that non-work messages have gro...,1,1,1,7.205144,70824b8f-d252-41e4-bc55-e323c4aca735,47f83a26-ef7f-4b27-9557-3e2b563d6c21
1,How do LLMs like ChatGPT's AI capabiltys help ...,"Alright, buckle up—here’s the inside scoop dri...",None,"ChatGPT, based on Large Language Models (LLMs)...",1,1,1,5.504997,bb8953d8-6be4-4177-9417-d1861ccc43e0,92991ccd-0e0a-4d97-a1b5-44f4f5818460
2,How do user behavior and usage patterns of Cha...,"Yo, here’s the scoop straight from the AI stre...",None,"The rapid adoption of ChatGPT, with 18 billion...",1,1,1,6.840715,9bfea33d-03a6-48d5-9f31-93c9d3f1b046,9ef0b048-438b-4e3d-8c42-bf91a2c281b8
3,How does the launch and rapid adoption of Chat...,"Alright, buckle up for some next-level insight...",None,"ChatGPT was launched in November 2022 and, by ...",1,1,1,4.134989,d7f76581-4410-41ef-aa78-96e2bb663040,e6fecc54-33b8-4195-af09-450bacecf79d
4,Wht work activites and mesage distrubtion by o...,"Alright, buckle up for a turbo-charged dive in...",None,The data presents variation in ChatGPT usage b...,1,1,1,9.152883,eee0be09-86c6-4ff3-ae12-86823fab718c,c2825e6e-8c6c-4d2d-b343-be5fc54a6f19
5,What are SOC2 codes 19?,"Oh yeah, diving into that SOC2 realm—SOC2 code...",None,The context does not provide a specific defini...,0,0,1,1.782887,f61ed56f-a224-43a2-b63e-baeef398b0ce,dec4e0a2-1cf1-4b72-9a74-b7e88a10b605
6,SOC2 codes 13 what are they and how they relat...,"Alright, buckle up for some SOC2 code wizardry...",None,Variation by Occupation Figure 23 reports resu...,1,1,1,5.193452,bd773135-76a2-44a7-98f8-2fd225a9348b,58099416-bd72-4020-a640-88dcabe95959
7,What is the significance of June 2024 in the c...,Boom! June 2024 is like the ignition point for...,None,"The report provides data ending on June 26th, ...",1,0,1,4.103548,70771476-d2e2-45af-abbf-591f9768deff,3ed89173-c266-4b04-90cc-284657c06db5
8,What significance does June 2025 hold in the c...,June 2025 is straight-up legendary in the Chat...,None,"The context reports data ending on June 26, 20...",1,1,1,4.886559,116483b7-96d1-4116-8176-6e01f58c1588,ddca4276-d0ac-462a-8d8c-2a9909311303
9,Korinek and Suh 2024 what they say about AI an...,"Yo, here’s the slick scoop straight from the c...",None,"The context mentions that Korinek and Suh, 202...",1,1,1,3.123065,82d17e62-0616-402a-b242-eaa72d0b606b,844a756e-6942-48c0-9a8a-753e90c51951


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

- Larger chunks keep more context -> boosts correctness & helpfulness.

- Better embeddings encode more detail -> retrieval more precise -> improves correctness & helpfulness.

- Dope prompt makes model speak in expected style -> matches evaluator expectation -> dopness jumps 0 -> 1.

### Dashboard

![](./images/dashboard.png)

### Test 1

![](./images/test1.png)

### Test 2

![](./images/test2.png)
